In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (81408, 2)
Test shape: (8128, 2)


In [ ]:
train_df.head()


,tweets,class
0,Be aware dirty step to get money #staylight ...,figurative
1,#sarcasm for #people who don't understand #diy...,figurative
2,@IminworkJeremy @medsingle #DailyMail readers ...,figurative
3,@wilw Why do I get the feeling you like games?...,figurative
4,-@TeacherArthurG @rweingarten You probably jus...,figurative


In [ ]:
train_df["label"] = train_df["class"].astype(str).apply(lambda x: 1 if x.strip().lower()=="figurative" else 0)
test_df["label"] = test_df["class"].astype(str).apply(lambda x: 1 if x.strip().lower()=="figurative" else 0)


texts_train = train_df["tweets"].values
labels_train = train_df["label"].values
texts_test = test_df["tweets"].values
labels_test = test_df["label"].values

In [ ]:
train_df.sample(10)

,tweets,class,label
30841,People looking to be unfaithful exposed becaus...,irony,0
41595,Watching the New York feed on NESN+ #ironic #B...,irony,0
72833,I love how my random fact posts are doing what...,sarcasm,0
70324,"@ShaunKing shocking, police have such an illus...",sarcasm,0
49338,TURKEY: Joint Declaration of Women’s + LGBTI O...,regular,0
43925,Advocate The Forgotten Lesbian Film Classic Th...,regular,0
68539,@phil827 how excited are you?! #sarcasm https...,sarcasm,0
29298,I hope the first time Russell Wilson bangs Cia...,irony,0
47850,SETI? Just hope that Milner &amp; Hawking (see...,regular,0
10120,You get with a guy that everybody wants ; then...,figurative,1


In [ ]:

stop_words = set(stopwords.words('english'))
def preprocess(text):
    text = text.lower()
    text = "".join(c for c in text if c.isalpha() or c.isspace())
    tokens = text.split()
    return ' '.join([word for word in tokens if word not in stop_words])


clean_texts = [preprocess(t) for t in texts_train]
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(clean_texts)
sequences = tokenizer.texts_to_sequences(clean_texts)
padded_sequences = pad_sequences(sequences, maxlen=50)

In [ ]:
print("Original sample:", train_df.loc[1, "tweets"])
print("Cleaned sample:", clean_texts[1])
print("padded_sequences:", padded_sequences[1])

Original sample: #sarcasm for #people who don't understand #diy #artattack http://t.co/rtyYmuDVUS
Cleaned sample: sarcasm people dont understand diy artattack httptcortyymudvus
padded_sequences: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    1    8   16  332 6188]


In [ ]:
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

In [ ]:
embedding_index = {}
with open('glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]  # first element is the word (key)
        vector = np.asarray(values[1:], dtype='float32')  # rest are numbers (embedding vector)
        embedding_index[word] = vector


embedding_matrix = np.zeros((10000, 100))
for word, i in tokenizer.word_index.items():
    if i >= 10000:
        break
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
